# 🚦 Traffic Anomaly — GANomaly & VAE Training on Colab

This notebook trains both the **GANomaly** and **VAE** appearance-anomaly models on a free Colab GPU, then downloads the checkpoints.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU` (free tier)
2. Run cells **top-to-bottom** in order

Checkpoints are saved to Google Drive so they survive session restarts.

## 1 · Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU, then reconnect."
    )

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2 · Mount Google Drive

Checkpoints and metrics CSVs will be saved here so they survive session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/traffic_models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive checkpoint directory: {DRIVE_DIR}')

## 3 · Clone the repository & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/DasonTio/traffic.git'
REPO_DIR = '/content/traffic'
BRANCH   = 'chore/fine-tune'   # branch that has the latest training scripts

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install Python dependencies
# --quiet suppresses the progress bars so the output stays readable
!pip install -q -r requirements.txt
print('✅ Dependencies installed')

## 3.5 · Upload Dataset to Google Drive (IMPORTANT)\n\nBecause the `dataset/sequences` folder is ignored by git, Colab doesn`t download it automatically.\n**Before running the next cell:**\n1. Zip your local dataset folder: `zip -r dataset.zip dataset/sequence*`\n2. Upload `dataset.zip` to your Google Drive inside the `traffic_models` folder.\n

In [ ]:
import zipfile\nimport os\n\ndataset_zip = f"{DRIVE_DIR}/dataset.zip"\nif os.path.exists(dataset_zip):\n    print(f"Found dataset.zip in Drive, extracting...")\n    with zipfile.ZipFile(dataset_zip, "r") as zip_ref:\n        zip_ref.extractall(REPO_DIR)\n    print("✅ Dataset extracted successfully!")\nelse:\n    print(f"❌ ERROR: {dataset_zip} not found!")\n    print("Please upload your dataset.zip to Google Drive first.")\n

## 4 · Approve training sequences

Marks every sequence in `dataset/sequence_review.csv` as `approved` so the trainers have data to work with.

In [ ]:
!python scripts/approve_all_sequences.py

## 5 · Training configuration

Adjust the hyperparameters here before running the training cells.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
EPOCHS     = 30      # recommended: 20-50 on Colab GPU
BATCH_SIZE = 64      # T4 has 15 GB VRAM — 64 fits comfortably
WORKERS    = 2       # DataLoader workers
DEVICE     = 'cuda'  # 'cuda' or 'cpu'

# GANomaly-specific
GAN_LR     = 2e-4

# VAE-specific
VAE_LR     = 1e-3
VAE_BETA   = 1.0     # KL weight; increase to 4.0 for beta-VAE disentanglement

# Class groups to train (change to a subset if you want to skip one)
GROUPS = ['car', 'bus', 'truck']

print('Configuration:')
print(f'  epochs={EPOCHS}, batch_size={BATCH_SIZE}, workers={WORKERS}, device={DEVICE}')
print(f'  GAN lr={GAN_LR} | VAE lr={VAE_LR} beta={VAE_BETA}')
print(f'  groups={GROUPS}')

## 6 · Train GANomaly

One checkpoint per vehicle class. Results are saved to `models/` and also copied to Google Drive.

In [ ]:
import subprocess, shutil, time
from pathlib import Path

def run_and_stream(cmd: str) -> int:
    """Run a shell command and stream its output live."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

ganomaly_results = {}

for group in GROUPS:
    ckpt = f'models/ganomaly_{group}.pt'
    print(f'\n{"="*60}')
    print(f' GANomaly — group: {group}')
    print(f'{"="*60}')

    t0 = time.time()
    rc = run_and_stream(
        f'python scripts/train_ganomaly.py '
        f'--group {group} '
        f'--epochs {EPOCHS} '
        f'--batch-size {BATCH_SIZE} '
        f'--workers {WORKERS} '
        f'--lr {GAN_LR} '
        f'--device {DEVICE} '
        f'--output {ckpt}'
    )

    elapsed = time.time() - t0
    status = '✅' if rc == 0 else f'❌ (exit {rc})'
    ganomaly_results[group] = {'status': status, 'minutes': elapsed / 60}
    print(f'\n{status} {group} finished in {elapsed/60:.1f} min')

    # Copy checkpoint + metrics to Drive
    if rc == 0 and Path(ckpt).exists():
        shutil.copy(ckpt, f'{DRIVE_DIR}/ganomaly_{group}.pt')
        csv_path = ckpt.replace('.pt', '.csv')
        if Path(csv_path).exists():
            shutil.copy(csv_path, f'{DRIVE_DIR}/ganomaly_{group}.csv')
        print(f'   → Saved to Drive')

print('\n── GANomaly training summary ──')
for g, r in ganomaly_results.items():
    print(f'  {g}: {r["status"]} ({r["minutes"]:.1f} min)')

## 7 · Train VAE

In [ ]:
vae_results = {}

for group in GROUPS:
    ckpt = f'models/vae_{group}.pt'
    print(f'\n{"="*60}')
    print(f' VAE — group: {group}')
    print(f'{"="*60}')

    t0 = time.time()
    rc = run_and_stream(
        f'python scripts/train_vae.py '
        f'--group {group} '
        f'--epochs {EPOCHS} '
        f'--batch-size {BATCH_SIZE} '
        f'--workers {WORKERS} '
        f'--lr {VAE_LR} '
        f'--device {DEVICE} '
        f'--output {ckpt}'
    )

    elapsed = time.time() - t0
    status = '✅' if rc == 0 else f'❌ (exit {rc})'
    vae_results[group] = {'status': status, 'minutes': elapsed / 60}
    print(f'\n{status} {group} finished in {elapsed/60:.1f} min')

    if rc == 0 and Path(ckpt).exists():
        shutil.copy(ckpt, f'{DRIVE_DIR}/vae_{group}.pt')
        csv_path = ckpt.replace('.pt', '.csv')
        if Path(csv_path).exists():
            shutil.copy(csv_path, f'{DRIVE_DIR}/vae_{group}.csv')
        print(f'   → Saved to Drive')

print('\n── VAE training summary ──')
for g, r in vae_results.items():
    print(f'  {g}: {r["status"]} ({r["minutes"]:.1f} min)')

## 8 · Full training summary

In [ ]:
from pathlib import Path

print('=' * 60)
print(' Training Complete — Final Summary')
print('=' * 60)

print('\nGANomaly checkpoints:')
for group in GROUPS:
    p = Path(f'models/ganomaly_{group}.pt')
    size = f'{p.stat().st_size / 1e6:.1f} MB' if p.exists() else 'NOT FOUND'
    r = ganomaly_results.get(group, {})
    print(f'  {group:<8} {r.get("status", "?")}  {size}')

print('\nVAE checkpoints:')
for group in GROUPS:
    p = Path(f'models/vae_{group}.pt')
    size = f'{p.stat().st_size / 1e6:.1f} MB' if p.exists() else 'NOT FOUND'
    r = vae_results.get(group, {})
    print(f'  {group:<8} {r.get("status", "?")}  {size}')

print(f'\nAll checkpoints also saved to Google Drive: {DRIVE_DIR}')

## 9 · Download checkpoints to your local machine

Run this cell to zip all six checkpoints and download them directly from Colab.
You can also find them in your **Google Drive → `traffic_models/`** folder.

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

zip_path = '/content/traffic_models.zip'
model_files = [
    *[f'models/ganomaly_{g}.pt' for g in GROUPS],
    *[f'models/vae_{g}.pt'      for g in GROUPS],
    *[f'models/ganomaly_{g}.csv' for g in GROUPS],
    *[f'models/vae_{g}.csv'      for g in GROUPS],
]

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in model_files:
        p = Path(fp)
        if p.exists():
            zf.write(p, p.name)
            print(f'  added: {p.name}')
        else:
            print(f'  skip (not found): {p}')

print(f'\nDownloading {zip_path} ...')
files.download(zip_path)

## 10 · (Optional) Copy checkpoints back into your local repo

After downloading `traffic_models.zip`, extract it and copy the `.pt` files into your local `models/` folder:

```bash
unzip traffic_models.zip -d /path/to/traffic/models/
```

Then run the full evaluation locally:

```bash
source .venv/bin/activate
python main.py --batch --source-mode youtube --max-frames 5000
python scripts/seed_appearance_ground_truth.py
python scripts/run_full_evaluation.py
```